# 06 — Execution and Performance Basics

**Environment:** Ubuntu 22.04 · Python 3.12.3 · OpenJDK 17 · Spark 4.1.x · local PySpark.

Run cells in order. Each notebook can start a Spark session independently.

**Learning goal:** Explain lazy evaluation, actions, stages, shuffles, partition count, caching and query plans. On this tiny dataset, timings are *not* representative of production performance.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data' / 'transactions.csv').is_file()), None)
assert ROOT is not None, 'Open the workshop folder in Jupyter before running this notebook.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from training_lib import create_spark, read_raw, typed_transactions, quality_condition, clean_transactions, completed_enriched, revenue_by_category
from pyspark.sql import functions as F
spark = create_spark('TG117-Workshop')
print('Spark:', spark.version, '| Python:', sys.version.split()[0], '| master:', spark.sparkContext.master)

In [ ]:
raw, customers, products = read_raw(spark, ROOT)
clean = clean_transactions(raw)
print('Input partitions:',raw.rdd.getNumPartitions())
print('Clean partitions:',clean.rdd.getNumPartitions())

### Lazy transformations and physical plan

In [ ]:
prepared = clean.filter(F.col('status')=='COMPLETED')
joined = prepared.join(products,'product_id','inner')
summary = joined.groupBy('category').agg(F.sum('line_total').alias('revenue'))
print('Plan before an action:')
summary.explain('formatted')
summary.show()  # action

### See partition changes after shuffle

In [ ]:
repartitioned = clean.repartition(4, 'product_id')
print('Partitions after repartition:', repartitioned.rdd.getNumPartitions())
repartitioned.groupBy(F.spark_partition_id().alias('partition_id')).count().orderBy('partition_id').show()

### Cache only when the same DataFrame is reused

Caching has memory cost and the first action still computes the data.

In [ ]:
reuse = clean.filter(F.col('status')=='COMPLETED').cache()
print('Materialize cache:', reuse.count())
reuse.groupBy('product_id').count().show(12)
reuse.unpersist()

**Checks:** Which operation likely causes a shuffle: `select`, `filter`, or `groupBy`? Is `cache()` an action? Why must we avoid claiming performance speedups based on a 58-row dataset?